# Nyoba lagi clusteringg

In [2]:
! pip install kmodes joblib


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#import library
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from kmodes.kmodes import KModes
from sklearn.metrics import silhouette_score

In [5]:
#path
DATA_PATH = Path("../dataset/skincare_preprocessed.csv")
OUTPUT_CLUSTERED_PATH = Path("../dataset/skincare_clustered.csv")
MODEL_DIR = Path("../models/clustering")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Output path:", OUTPUT_CLUSTERED_PATH)
print("Model dir:", MODEL_DIR)

Data path: ..\dataset\skincare_preprocessed.csv
Output path: ..\dataset\skincare_clustered.csv
Model dir: ..\models\clustering


In [ ]:
#load data
df = pd.read_csv(DATA_PATH)

print("Jumlah data:", len(df))
print("Jumlah kolom:", len(df.columns))
df.head()

Jumlah data: 9848
Jumlah kolom: 21


,brand,name,type,ingridients,Acne_Fighting,Acne_Trigger,Anti_Aging,Brightening,Dark_Spots,Drying,...,Hydrating,Irritating,Large_Pores,Oily,Redness,Reduces_Irritation,Rosacea,Scar,Skin_Texture,Worsen_Oily
0,The Ordinary,Glycolic Acid 7% Toning Solution,Toner,"Water,Glycolic Acid,Rosa Damascena Flower Wate...",0,1,1,1,1,1,...,0,1,1,1,0,0,1,0,1,0
1,La Roche-Posay,Toleriane Hydrating Gentle Face Cleanser,Face Cleanser,"Water,Glycerin,Pentaerythrityl Tetraethylhexan...",1,1,1,1,0,1,...,0,0,0,1,1,1,0,0,0,0
2,Youth To The People,Superfood Antioxidant Cleanser,Face Cleanser,"Water,Cocamidopropyl Hydroxysultaine,Sodium Co...",0,1,1,1,1,1,...,0,1,1,0,1,1,0,0,1,0
3,COSRX,Low pH Good Morning Gel Cleanser,Face Cleanser,"Water,Cocamidopropyl Betaine,Sodium Lauroyl Me...",0,1,0,0,0,1,...,0,1,1,1,0,1,0,0,0,0
4,La Roche-Posay,Toleriane Purifying Foaming Face Cleanser,Face Cleanser,"Water,Glycerin,Coco-Betaine,Propylene Glycol,S...",1,1,1,1,0,1,...,0,0,0,1,1,0,0,0,0,0


In [ ]:
#cek tipe data
df["type"].value_counts()

type
Moisturizer       2749
Serum             2199
Face Cleanser     1689
Sunscreen         1277
Toner              865
Exfoliator         549
Makeup Remover     520
Name: count, dtype: int64

In [10]:
#atur fitur
# fitur tambahan
feature_cols = [
    "Oily",
    "Acne_Fighting",
    "Redness",
    "Large_Pores",
    "Skin_Texture",
    "Dark_Spots",
    "Scar",
    "Anti_Aging",
    "Brightening",
    "Hydrating",
    "Eczema",
    "Rosacea"
]

# tambahan fitur tambahan sebagai informasi
info_feature_cols = [
    "Worsen_Oily",
    "Acne_Trigger",
    "Irritating",
    "Drying",
    "Reduces_Irritation"
]


In [11]:
#cek semua fitur
missing_cols = [col for col in feature_cols if col not in df.columns]

if missing_cols:
    print("Kolom yang belum ada:", missing_cols)
else:
    print("Semua fitur clustering tersedia.")

Semua fitur clustering tersedia.


In [12]:
# fungsi file aman 
def safe_filename(text):
    return text.lower().replace(" ", "_").replace("/", "_")

In [ ]:
# cari k terbaik
def find_best_k_kmodes(X, k_range=range(2, 11)):
    best_score = -1
    best_k = None
    best_model = None
    best_labels = None
    results = []

    for init_method in ["Huang", "Cao"]:
        print(f"\nInit method: {init_method}")

        for k in k_range:
            kmodes = KModes(
                n_clusters=k,
                init=init_method,
                n_init=20,
                random_state=42,
                verbose=0
            )

            labels = kmodes.fit_predict(X)

            if len(set(labels)) < 2:
                continue

            score = silhouette_score(X, labels, metric="hamming")
            cluster_counts = pd.Series(labels).value_counts()
            min_cluster_size = cluster_counts.min()

            results.append({
                "init": init_method,
                "k": k,
                "silhouette_score": score,
                "min_cluster_size": min_cluster_size
            })

            print(f"K={k} | Score={score:.4f} | Min cluster={min_cluster_size}")

            if score > best_score:
                best_score = score
                best_k = k
                best_model = kmodes
                best_labels = labels

    results_df = pd.DataFrame(results)

    return best_k, best_score, best_model, best_labels, results_df

In [14]:
#clustering per type
type_list = [
    "Serum",
    "Moisturizer",
    "Face Cleanser",
    "Sunscreen",
    "Toner",
    "Exfoliator",
    "Makeup Remover"
]

all_clustered_data = []
cluster_summary = []
all_k_results = []

for type_name in type_list:
    print("\n" + "="*80)
    print(f"CLUSTERING TYPE: {type_name}")
    print("="*80)

    df_type = df[df["type"] == type_name].copy()

    if len(df_type) == 0:
        print(f"Tidak ada data untuk type: {type_name}")
        continue

    X = df_type[feature_cols].astype(int)

    print(f"Jumlah produk: {len(df_type)}")

    best_k, best_score, best_model, best_labels, k_results_df = find_best_k_kmodes(X)

    df_type["cluster"] = best_labels
    all_clustered_data.append(df_type)

    k_results_df["type"] = type_name
    all_k_results.append(k_results_df)

    model_package = {
        "model": best_model,
        "type": type_name,
        "features": feature_cols,
        "best_k": best_k,
        "silhouette_score": best_score
    }

    model_path = MODEL_DIR / f"kmodes_{safe_filename(type_name)}.pkl"
    joblib.dump(model_package, model_path)

    cluster_summary.append({
        "type": type_name,
        "jumlah_data": len(df_type),
        "best_k": best_k,
        "silhouette_score": round(best_score, 4),
        "model_path": str(model_path)
    })

    print("\nHASIL TERBAIK")
    print("Best K:", best_k)
    print("Best Silhouette Score:", round(best_score, 4))
    print("Model tersimpan di:", model_path)

    print("\nDistribusi cluster:")
    print(df_type["cluster"].value_counts().sort_index())


CLUSTERING TYPE: Serum
Jumlah produk: 2199

Init method: Huang
K=2 | Score=0.3245 | Min cluster=1078
K=3 | Score=0.3094 | Min cluster=486
K=4 | Score=0.3786 | Min cluster=384
K=5 | Score=0.3951 | Min cluster=288
K=6 | Score=0.3775 | Min cluster=145
K=7 | Score=0.3699 | Min cluster=115
K=8 | Score=0.3702 | Min cluster=92
K=9 | Score=0.3592 | Min cluster=84
K=10 | Score=0.3622 | Min cluster=57

Init method: Cao
K=2 | Score=0.2874 | Min cluster=771
K=3 | Score=0.3046 | Min cluster=504
K=4 | Score=0.3666 | Min cluster=440
K=5 | Score=0.3079 | Min cluster=60
K=6 | Score=0.3029 | Min cluster=60
K=7 | Score=0.3218 | Min cluster=58
K=8 | Score=0.3499 | Min cluster=78
K=9 | Score=0.3032 | Min cluster=67
K=10 | Score=0.3122 | Min cluster=67

HASIL TERBAIK
Best K: 5
Best Silhouette Score: 0.3951
Model tersimpan di: ..\models\clustering\kmodes_serum.pkl

Distribusi cluster:
cluster
0    448
1    288
2    408
3    735
4    320
Name: count, dtype: int64

CLUSTERING TYPE: Moisturizer
Jumlah produk: 

In [15]:
#menggabungkan hasil
df_clustered = pd.concat(all_clustered_data, ignore_index=True)

df_clustered.to_csv(OUTPUT_CLUSTERED_PATH, index=False)

cluster_summary_df = pd.DataFrame(cluster_summary)
cluster_summary_df.to_csv(MODEL_DIR / "cluster_summary.csv", index=False)

k_results_all_df = pd.concat(all_k_results, ignore_index=True)
k_results_all_df.to_csv(MODEL_DIR / "all_k_results.csv", index=False)

print("Clustering selesai.")
print("Dataset hasil clustering tersimpan di:", OUTPUT_CLUSTERED_PATH)
print("Ringkasan clustering tersimpan di:", MODEL_DIR / "cluster_summary.csv")
print("Hasil percobaan semua K tersimpan di:", MODEL_DIR / "all_k_results.csv")

display(cluster_summary_df)

Clustering selesai.
Dataset hasil clustering tersimpan di: ..\dataset\skincare_clustered.csv
Ringkasan clustering tersimpan di: ..\models\clustering\cluster_summary.csv
Hasil percobaan semua K tersimpan di: ..\models\clustering\all_k_results.csv


,type,jumlah_data,best_k,silhouette_score,model_path
0,Serum,2199,5,0.3951,..\models\clustering\kmodes_serum.pkl
1,Moisturizer,2749,5,0.3909,..\models\clustering\kmodes_moisturizer.pkl
2,Face Cleanser,1689,4,0.4567,..\models\clustering\kmodes_face_cleanser.pkl
3,Sunscreen,1277,4,0.4153,..\models\clustering\kmodes_sunscreen.pkl
4,Toner,865,4,0.4166,..\models\clustering\kmodes_toner.pkl
5,Exfoliator,549,7,0.4809,..\models\clustering\kmodes_exfoliator.pkl
6,Makeup Remover,520,5,0.5264,..\models\clustering\kmodes_makeup_remover.pkl


In [16]:
#validasi hasil clustering
print("Jumlah data hasil clustering:", len(df_clustered))
print("Jumlah cluster kosong/null:", df_clustered["cluster"].isna().sum())

print("\nJumlah produk per type:")
display(df_clustered["type"].value_counts())

print("\nJumlah cluster per type:")
display(df_clustered.groupby("type")["cluster"].nunique())

Jumlah data hasil clustering: 9848
Jumlah cluster kosong/null: 0

Jumlah produk per type:


type
Moisturizer       2749
Serum             2199
Face Cleanser     1689
Sunscreen         1277
Toner              865
Exfoliator         549
Makeup Remover     520
Name: count, dtype: int64


Jumlah cluster per type:


type
Exfoliator        7
Face Cleanser     4
Makeup Remover    5
Moisturizer       5
Serum             5
Sunscreen         4
Toner             4
Name: cluster, dtype: int64

In [17]:
#karakteristik cluster
for type_name in type_list:
    print("\n" + "="*80)
    print(f"PROFIL CLUSTER: {type_name}")
    print("="*80)

    df_type = df_clustered[df_clustered["type"] == type_name].copy()

    if len(df_type) == 0:
        continue

    cluster_profile = df_type.groupby("cluster")[feature_cols].mean().round(2)

    for cluster_id in cluster_profile.index:
        print(f"\nCluster {cluster_id}")
        top_tags = cluster_profile.loc[cluster_id].sort_values(ascending=False).head(6)
        print(top_tags)

        print("\nContoh produk:")
        contoh_produk = df_type[df_type["cluster"] == cluster_id][["brand", "name", "type"]].head(5)
        print(contoh_produk.to_string(index=False))


PROFIL CLUSTER: Serum

Cluster 0
Large_Pores    0.35
Anti_Aging     0.23
Eczema         0.23
Rosacea        0.22
Redness        0.21
Brightening    0.19
Name: 0, dtype: float64

Contoh produk:
        brand                                    name  type
  KIKO Milano                      Smart Charge Drops Serum
      Clarins    Super Restorative Remodelling Serum  Serum
   La Prairie Platinum Rare Haute-Rejuvenation Elixir Serum
     May Coop          Vine Expert Serum "Anti-Aging" Serum
Dr Botanicals        Kiwi Superfood Cooling Eye Serum Serum

Cluster 1
Anti_Aging      0.98
Brightening     0.97
Scar            0.86
Oily            0.35
Skin_Texture    0.31
Redness         0.29
Name: 1, dtype: float64

Contoh produk:
        brand                                   name  type
 The Ordinary            Azelaic Acid Suspension 10% Serum
   StriVectin Peptight™ Tightening Neck Serum Roller Serum
   La Prairie          Skin Caviar Liquid Lift Serum Serum
Ikaria Beauty             Daytime

In [18]:
#cek file pkl sudah tersimpan belum
print("File model clustering yang tersimpan:")

for file in MODEL_DIR.glob("*.pkl"):
    print(file)

File model clustering yang tersimpan:
..\models\clustering\kmodes_exfoliator.pkl
..\models\clustering\kmodes_face_cleanser.pkl
..\models\clustering\kmodes_makeup_remover.pkl
..\models\clustering\kmodes_moisturizer.pkl
..\models\clustering\kmodes_serum.pkl
..\models\clustering\kmodes_sunscreen.pkl
..\models\clustering\kmodes_toner.pkl
